# SECOM dataset ML Project
---

### Loading in the SECOM dataset

In [ ]:
# Imports
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif

In [ ]:
# Pulling in the SECOM data and loading data into feature and label dataframes
secom = fetch_ucirepo(id=179)
df = pd.DataFrame(secom.data.original)
X = df.drop(columns=["class", "timestamp"])
y = df["class"]

print(X.shape)
print(y.shape)

---

### Exploratory Data Analysis (EDA)

In [ ]:
# checking data types of features
X.dtypes.value_counts()

In [ ]:
# checking null counts and null percentage of features
feature_summary = pd.DataFrame({
    "null_counts": X.isna().sum(),
    "null_pct": round(X.isna().mean()*100, 2)
}).sort_values("null_pct", ascending=False)

print(feature_summary.head(50))

In [ ]:
# checking class balance and null counts
label_summary = pd.DataFrame({
    "counts": y.value_counts(),
    "null_counts": y.isna().sum(),
    "pct": round(y.value_counts()/len(y)*100, 2),    
})

print(label_summary)

---

In [ ]:
# filtering out features with null percentage above threshold
threshold = 90 # null pct threshold

null_cols = X.columns[X.isna().mean()*100 > threshold]
X_filtered = X.drop(columns=null_cols)
X_filtered.shape

In [ ]:
# Fill remaining NaN values with median
X_filtered = X_filtered.fillna(X_filtered.median())
X_filtered.head()

In [ ]:
# Remove constant variance terms
zero_var_cols = X_filtered.columns[X_filtered.var() == 0]
X_filtered = X_filtered.drop(columns=zero_var_cols)
X_filtered.shape

In [ ]:
# Checking variance threshold to see if we have an obvious cutoff point
variances = X_filtered.var()

plt.hist(np.log10(variances), bins=50, color='#4B7F68')
plt.xlabel("log10(Variance)")
plt.ylabel("Count")
plt.grid()
plt.show()

In [ ]:
# running a correlation analysis to identify highly correlated features
corr = X_filtered.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().reset_index()
pairs.columns = ["feature_1", "feature_2", "corr"]
pairs = pairs.sort_values(
    "corr",
    ascending=False
)

print(pairs.head(50))

In [ ]:
# identifying columns to drop that are correlated more than 95%
to_drop = [
    col
    for col in upper.columns
    if any(upper[col] > 0.95)
]

print(f"Columns to drop: {len(to_drop)}")

In [ ]:
# examining the highly correlated feature pairs
high_corr = pairs[pairs["corr"] > 0.95]
high_corr.sort_values("corr", ascending=False).head(100)

In [ ]:
# removing the columns correlated above threshold from the dataset
X_uncorr = X_filtered.drop(columns=to_drop)
X_uncorr.shape

In [ ]:
# checking correlations with target
mi = mutual_info_classif(
    X_uncorr,
    y,
    random_state=42
)

mi = pd.Series(
    mi,
    index=X_uncorr.columns
)

mi.sort_values(
    ascending=False
).describe()

In [ ]:
# checking how many features have zero mutual info with target
zero_mi = mi.value_counts()
zero_mi

In [ ]:
# plotting histogram of log mi to see distribution
plt.hist(np.log10(mi[mi > 0]), bins=50, color='#4B7F68')
plt.xlabel("log10(mi)")
plt.ylabel("counts")

___

### Final Remarks
1. classes highly imbalanced
2. features highly correlated
3. No obvious variance cutoff after removing zero-variance features
4. Mutual information is not very high for any given sensor, will not use MI to exclude features

### Final Preprocessing Recommendation
1. Remove null features with > 90% nulls
2. Impute missing data with median
3. Remove zero-variance features
4. Drop columns with > 95% correlation

### Modeling Suggestions
1. Try logistic regression with L1 regularization and inspect feature coefficients
2. Try PLS-DA and check latent variables for key features
3. Run XGBoost and compare performance gains with the linear models

